In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print("Environment variables are being loaded")

Environment variables are being loaded


In [52]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

In [5]:
## Custom Pre-Processor Function to Process the PDF
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],

        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Laod PDF

        loader=PyMuPDFLoader(pdf_path)
        pages=loader.load()

        # Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            # clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

In [6]:
preprocessor=SmartPDFProcessor()

In [7]:
preprocessor

In [8]:
## Process a PDF with Pre-Processor Function and Create Chunks
try:
    smart_chunks=preprocessor.process_pdf("Data\FSS_Gazete_Rules_2011 comendium.pdf")  # document path
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

<>:3: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<>:3: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
C:\Users\Nikshita Chilivari\AppData\Local\Temp\ipykernel_12652\3142564519.py:3: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
  smart_chunks=preprocessor.process_pdf("Data\FSS_Gazete_Rules_2011 comendium.pdf")  # document path


Processed into 115 smart chunks

Sample chunk metadata:
  producer: Microsoft® Word 2016
  creator: Microsoft® Word 2016
  creationdate: 2025-04-03T15:04:57+05:30
  source: Data\FSS_Gazete_Rules_2011 comendium.pdf
  file_path: Data\FSS_Gazete_Rules_2011 comendium.pdf
  total_pages: 46
  format: PDF 1.5
  title: 
  author: DELL
  subject: 
  keywords: 
  moddate: 2025-04-03T15:04:57+05:30
  trapped: 
  modDate: D:20250403150457+05'30'
  creationDate: D:20250403150457+05'30'
  page: 1
  chunk_method: smart_pdf_processor
  char_count: 2225


In [9]:
# Print the chunks
print(smart_chunks[0])
print("-----------------------------")
print(smart_chunks[5])
print("-----------------------------")
print(smart_chunks[11])
print("-----------------------------")
print(smart_chunks[41])

page_content='Version I, 01.04.2025 Food Safety and Standards Rules, 2011 1.1.1: These rules may be called the Food Safety and Standards Rules, 2011. 1.2: Definitions 1.2.1: In these rules, unless the context otherwise requires, 1. “Act” means the Food Safety and Standards Act, 2006 (Act 34 of 2006) ; 2. “Adjudicating Officer” means the Adjudicating Officer appointed under sub-section (1) of section68 of the Act. 3. “Advocate” means a person who is entitled to practice the profession of law under the Advocates Act, 1961 (25 of 1961) 4. “Appellate Tribunal” means the Food Safety Appellate Tribunal constituted under section 70 of the Act. 5. “Authorised Officer” means an officer authorized by the Food Authority referred in the sub-section (5) of section 47 of the Act. 6. “Inquiry” means the inquiry referred to in section 68. 7. “Licensing Authority” means the Designated Officer appointed under section 36 of the Act for the local area and includes any other officer so appointed for the pu

# Initialize Huggingface Embeddings and Create FAISS Vector Store

In [10]:
## Import required modules (LATEST imports)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
## Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",   # Huggingface embeddings model
    model_kwargs={"device": "cpu"},                        # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2806.16it/s]


In [12]:
## Create FAISS Vector Store from smart_chunks
vectorstore = FAISS.from_documents(
    documents=smart_chunks,
    embedding=embeddings
)
print(f"\n Embeddings are created and stored in memory.")


 Embeddings are created and stored in memory.


In [13]:
## Save FAISS Index to Local Disk

FAISS_INDEX_PATH = "faiss_index"

vectorstore.save_local(FAISS_INDEX_PATH)

print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [14]:
## Load FAISS Index Later

loaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 115 vectors


In [15]:
## Quick Similarity Test
query = "What is this document about?"
docs = loaded_vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])


--- Result 1 ---
on _______ and in confirmation thereof, I have initialed each page of such extracts/copies. 7. I am aware that based on the solemn declarations given in this Affidavit, the Food Safety Officer has agreed to return the books of accounts and other documents seized as aforesaid on __________. 8. I here

--- Result 2 ---
otherwise provided hereunder, discovery or production and return of documents shall be regulated by the provisions of Civil Procedure Code, 1908. (ii) An application for summons to produce documents shall set out the document/s production of which is sought; the relevancy of the document/s and in ca

--- Result 3 ---
Version I, 01.04.2025 1. The Registrar shall discharge his functions under the general superintendence of the Presiding Officer. He shall discharge such other functions as are assigned to him under these rules by the Presiding Officer. 2. The Registrar shall have the custody of the records of the Ap


# Define LLM, Prompt Template and Retriever

In [16]:
load_dotenv()    # Load environment

True

In [72]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")            # Load API Key

In [59]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY")
)

print(llm.invoke("What is LangChain?").content)

## What is LangChain?

**LangChain** is an open‑source framework that helps developers build applications powered by large language models (LLMs) such as GPT‑4, Claude, Gemini, Llama‑2, etc.  
It abstracts away the plumbing needed to combine LLM calls, prompts, memory, external tools, and data sources into a single, composable “chain” of operations.

---

### Core Idea

- **Chain** – A sequence of operations (prompts, LLM calls, function calls, etc.) that can be composed, reused, and debugged.
- **Memory** – Persisting context across turns so that the model can “remember” earlier parts of a conversation or task.
- **Tools/Agents** – Plug‑in interfaces that let an LLM call APIs, run Python code, query databases, etc., as if it were a human “tool‑using” agent.
- **Retrieval** – Automatic retrieval of relevant documents or knowledge from a vector store before feeding them to the LLM.

---

### Key Components

| Component | Purpose | Typical Usage |
|-----------|---------|---------------|


In [60]:
llm.invoke("Explain what is LangChain in one paragraph.")  # Test Query to LLM

AIMessage(content='LangChain is an open‑source framework that streamlines the development of applications powered by large language models (LLMs). It abstracts the complexity of interacting with LLM APIs by providing reusable building blocks—such as prompt templates, chains, memory, and agents—so developers can compose, orchestrate, and fine‑tune multi‑step workflows that combine LLM inference with external data sources, APIs, and custom logic. By handling token limits, context management, and integration patterns, LangChain lets teams rapidly prototype and deploy sophisticated conversational, analytical, or generative tools without reinventing the core LLM interaction layer.', additional_kwargs={'reasoning_content': "We need to explain LangChain in one paragraph. It's a framework for building LLM applications, provides components like prompt templates, chain, memory, agents, etc. Provide concise explanation. Let's do."}, response_metadata={'token_usage': {'completion_tokens': 174, 'pr

In [61]:
# Define Prompt Template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [62]:
# Define Retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [63]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000254551EFE00>, search_kwargs={'k': 3})

# Create RAG Chain using Langchain Expression Language(LCEL) and Query the RAG Chain

In [64]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [65]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Create RAG Chain

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [66]:
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000254551EFE00>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False,

In [67]:
response = rag_chain.invoke(
    "What are the powers and duties of a Designated Officer?"
)

print(response)


The powers and duties of a Designated Officer are those prescribed in **Section 36(3) of the Food Safety and Standards Act, 2006**.  In practice, the officer exercises those powers while working under the overall supervision of the District Collector (or District Magistrate).


In [68]:
response = rag_chain.invoke("What steps must a Food Safety Officer follow while collecting a food sample?")
print(response)

**Steps a Food Safety Officer must follow when collecting a food sample**

1. **Prepare the sampling kit**  
   - Use the prescribed type of container and seal.  
   - Maintain the required temperature and transportation conditions as directed by the Food Authority.

2. **Notify the Food Analyst**  
   - Inform the Analyst of the sample lift, including method, container, and conditions to preserve integrity.

3. **Call witnesses**  
   - Call one or more witnesses at the time of lifting the sample.

4. **Obtain signatures**  
   - Secure the witnesses’ signatures on all forms and documents prepared for the sampling.

5. **Serve the notice**  
   - Serve a notice in Form V A to the business operator immediately at the time of sampling.

6. **Notify related parties** (if applicable)  
   - If the product is supplied by a manufacturer, distributor or supplier, give notice to that party as well.

7. **Sample from an open container**  
   - If the sample is taken from an open container, als

In [69]:
response = rag_chain.invoke("Explain the procedure for taking food samples for analysis.")
print(response)

**Procedure for taking food samples for analysis**

| Step | Action | Who performs it | Key points from the rules |
|------|--------|------------------|---------------------------|
| 1 | **Identify the sample** | Food Safety Officer (FSO) or Authorized Officer | The sample is taken for the purpose of analysis under the Act, except when it is for microbiological testing. |
| 2 | **Preserve the sample (if required)** | FSO |  • A preservative may be added to the sample **only** when the analysis is not microbiological. <br>• The preservative’s type and quantity must be written on the label that is affixed to the container (Rule 13). |
| 3 | **Maintain original packaging** | FSO |  • For foods sold in a sealed container or package, the sample should be sent **unopened** (as far as practicable) in its original condition, with an approximate quantity and the original label attached (Rule 14). |
| 4 | **Handle bulk packages** | FSO |  • When a bulk package is sampled, preservatives are added

In [70]:
response = rag_chain.invoke("Summarize Chapter 3 of the Food Safety and Standards Rules, 2011.")
print(response)

**Chapter 3 – Food Safety Surveillance, Inspection and Enforcement (2011 Rules)**  

Chapter 3 establishes the framework for ongoing monitoring and enforcement of food safety standards across India. Its key provisions are:  

1. **Surveillance & Inspection** – Food Safety Officers (FSOs) are required to conduct surveillance in accordance with plans issued by the Central or State Food Authority. They must inspect vehicles, premises, and imported consignments suspected of carrying unsafe or non‑compliant food.  

2. **Training & Authority** – Officers appointed under clauses 2 and 3 of the chapter must receive specialised training within five years of the rules’ commencement. FSOs are empowered, under section 38 of the Act, to seize adulterated or sub‑standard food, seal premises for investigation, and take samples for analysis when compliance cannot be achieved.  

3. **Regulatory Actions** – FSOs may recommend corrective measures to the Designated Officer, including revocation or modif

In [71]:
for chunk in rag_chain.stream("Summarize the document"):
    print(chunk, end="", flush=True)

**Summary of the Extracted Sections**

1. **Food Analyst Duties (Document 1)**
   - The Food Analyst must analyze the food article sent to them, following the Food Authority’s current instructions and procedures.  
   - The analysis report must be signed by the Analyst.  
   - Upon completion, the report is sent to the designated officer or purchaser in Form VII A.

2. **Appeal/ Application Procedures (Document 2)**
   - A separate memorandum of appeal is unnecessary if the interim order or direction is already requested in the main appeal.  
   - Appeals must be in English or the state’s local language, type‑written or printed, double‑spaced, on one side of standard petition paper, paginated, indexed, and bound.  
   - Content must be divided into numbered paragraphs.  
   - Any changes (erasing, correcting, deleting) to a filed appeal must be initialed by the party or their authorized agent.

3. **Memorandum of Appeal Submission & Content (Document 3)**
   - A memorandum sent by regi